# **Features Selection - Xây dựng các đặc trưng cho mô hình**

Sau khi đã khám phá dữ liệu và những phương pháp huấn luyện mô hình đang có, trong phần này chúng tôi sẽ sàng lọc và cung cấp thêm những features quan trọng khác cho bộ dữ liệu huấn luyện của chúng tôi dựa trên đữ liệu cơ sở `base.csv`. Chúng tôi sẽ xây dựng tất cả các features có liên quan tới 2 phương pháp để dùng cho các bước sau

Cụ thể:
- Trích xuất toàn bộ dữ liệu chung hữu ích
- Xây dựng 2 bộ dữ liệu với 2 phương pháp khác nhau
- Tính toán, xây dựng các đặc trưng hữu ích cho từng bộ dữ liệu

**Mục tiêu**: Xây dựng lại tập huấn luyện với nhiều đặc trưng để cải thiện mô hình dự đoán

Mục lục:
1. [Thiết lập và cài đặt](#1)
2. [Xây dựng bộ dữ liệu cơ sở](#2)
3. [Dữ liệu cho dự đoán trực tiếp](#3)
4. [Dữ liệu cho dự đoán gián tiếp](#4)

<a id='1'></a>

## 1. Thiết lập và cài đặt

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import sys, os
sys.path.append(os.path.abspath('..'))
from src.get_data import get_connection, get_data_processed
from src.save_data import save_to_processed

In [23]:
conn = get_connection()

[OKE] Kết nối thành công tới database tại D:\datathon\vimchanhxa-datathon\data\database\datathon.duckdb


In [3]:
df_base = get_data_processed('base.csv')
df_base['date'] = pd.to_datetime(df_base['date'])
df_base.head()

Đã đọc thành công dữ liệu từ: D:\datathon\vimchanhxa-datathon\data\processed\base.csv


,date,revenue,cogs,is_test,order_count,unique_customers,total_quantity
0,2012-07-04,5123547.94,3982991.19,0,162.0,161.0,777.0
1,2012-07-05,2751773.45,2150580.23,0,97.0,97.0,428.0
2,2012-07-06,3054029.42,2517632.84,0,93.0,93.0,441.0
3,2012-07-07,2667930.94,2108246.62,0,73.0,73.0,364.0
4,2012-07-08,2360851.90,1808622.79,0,88.0,87.0,394.0


In [4]:
print(f"Train Period: {df_base[df_base['is_test'] == 0]['date'].min().date()} to {df_base[df_base['is_test'] == 0]['date'].max().date()}")
print(f"Test Period : {df_base[df_base['is_test'] == 1]['date'].min().date()} to {df_base[df_base['is_test'] == 1]['date'].max().date()}")

Train Period: 2012-07-04 to 2022-12-31
Test Period : 2023-01-01 to 2024-07-01


## 2. Xây dựng bộ cơ sở dữ liệu

Trong phần này, chúng tôi sẽ trích xuất ra những features quan trọng, đặc biệt là thời gian và các dữ liệu tĩnh có tính lặp lại

### 2.1 Thời gian

In [5]:
df_base["day_of_week"] = df_base["date"].dt.dayofweek
df_base["day_of_month"] = df_base["date"].dt.day
df_base["day_of_year"] = df_base["date"].dt.dayofyear
df_base["month"] = df_base["date"].dt.month
df_base["year"] = df_base["date"].dt.year
df_base["is_weekend"] = np.where(df_base["day_of_week"] >= 5, 1, 0)
df_base["days_since_start"] = (df_base["date"] - df_base["date"].min()).dt.days

### 2.2 Sự kiện đặc biệt

In [6]:
# Sự kiện những ngày nhận lương, thường là từ 25 đến 5 hàng tháng, sẽ có nhu cầu mua sắm cao hơn
df_base["is_payday"] = np.where((df_base["day_of_month"] >= 25) | (df_base["day_of_month"] <= 5), 1, 0)

# Sự kiện những ngày có ngày trong tháng trùng với tháng (ví dụ 1/1, 2/2, ..., 12/12) có thể tạo ra những dịp mua sắm đặc biệt
df_base["is_double_day"] = np.where(df_base["month"] == df_base["day_of_month"], 1, 0)

In [7]:
# Những dịp tết âm lịch
tet_dates = {
    2012: "2012-01-23",
    2013: "2013-02-10",
    2014: "2014-01-31",
    2015: "2015-02-19",
    2016: "2016-02-08",
    2017: "2017-01-28",
    2018: "2018-02-16",
    2019: "2019-02-05",
    2020: "2020-01-25",
    2021: "2021-02-12",
    2022: "2022-02-01",
    2023: "2023-01-22",
    2024: "2024-02-10",
}
tet_map = pd.DataFrame(list(tet_dates.items()), columns=["year", "tet_date"])
tet_map["tet_date"] = pd.to_datetime(tet_map["tet_date"])

df_base = df_base.merge(tet_map, on="year", how="left")
df_base["days_to_tet"] = (df_base["tet_date"] - df_base["date"]).dt.days
df_base["is_pre_tet_rush"] = np.where((df_base["days_to_tet"] > 0) & (df_base["days_to_tet"] <= 21), 1, 0)
df_base["is_tet_holiday"] = np.where((df_base["days_to_tet"] <= 0) & (df_base["days_to_tet"] >= -6), 1, 0)
df_base = df_base.drop(columns=["tet_date"])

### 2.3 Mã giảm giá

In [29]:
promotions = conn.execute("SELECT * FROM promotions").df()
promotions.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   promo_id             50 non-null     str           
 1   promo_name           50 non-null     str           
 2   promo_type           50 non-null     str           
 3   discount_value       50 non-null     float64       
 4   start_date           50 non-null     datetime64[us]
 5   end_date             50 non-null     datetime64[us]
 6   applicable_category  10 non-null     str           
 7   promo_channel        50 non-null     str           
 8   stackable_flag       50 non-null     int32         
 9   min_order_value      50 non-null     float64       
dtypes: datetime64[us](2), float64(2), int32(1), str(5)
memory usage: 3.8 KB


Như ở phần `EDA`, chúng ta đã thấy rằng doanh nghiệp hiện tại đang mở các đợt giảm giá theo chu kỳ lặp lại hằng năm và hai năm. Cụ thể là với các mã giảm giá `Spring Sale`, `Mid-Year Sale`, `Fall Launch`, `Year-End Sale` đang được mở lại qua từng năm, còn với `Urban Blowout` và `Rural Special` thì lại lặp 2 năm một lần ( các năm lẻ ). Đặc biệt hơn là thời gian bắt đầu và kết thúc cũng rất khớp nhau, cùng lắm chỉ lệch 1 ngày. Tuy nhiên có một vài biến trong đó sẽ không được lặp lại giống hệt các năm trước như `min_order_value`, vì vậy dưới đây ta sẽ làm lại dữ liệu promotions này.

Do vẫn còn sự khác nhau, nên chúng tôi sẽ tạo thêm 2 cột đặc trưng để tránh bị loãng mô hình bao gồm: `is_promo` và `total_discount`, những ngày giảm giá và phần trăm đang giảm giá. Mục đích là hạn chế tạo quá nhiều cột nhiễu không cần thiết và vẫn thể hiện được sự khác biệt so với các mã giảm giá và các ngày khác. Sau đó, nếu `total_discount` không thể hiện được rõ vai trò, chúng tôi sẽ xóa đi vì trong dữ liệu sẽ có một vài mã có `stackable_flag` là không cho sử dụng 2 mã cùng lúc.

In [9]:
# Lọc các mã giảm giá lặp lại
promo_range = pd.date_range(start=df_base["date"].min(), end=df_base["date"].max())

daily_p = pd.DataFrame({
    "date": promo_range,
    "is_promo": 0,
    "total_discount": 0.0
})

for _, row in promotions.iterrows():
    mask = (
        (daily_p["date"] >= row["start_date"]) &
        (daily_p["date"] <= row["end_date"])
    )
    daily_p.loc[mask, "is_promo"] = 1
    daily_p.loc[mask, "total_discount"] += row["discount_value"]

df_base = (df_base.merge(daily_p, on="date", how="left").fillna({"is_promo": 0, "total_discount": 0}))

## 3. Dữ liệu cho dự đoán trực tiếp
Để dự đoán `revenue` trực tiếp, chúng tôi sẽ tạo ra những features khác hữu ích cho mô hình. Bao gồm các biến động từ quá khứ

## 4. Dữ liệu cho dự đoán gián tiếp

In [10]:
save_to_processed(df_base, "final_features.csv")

Đã lưu thành công tại: C:\Users\YOGA\Desktop\MyProjects\datathon\github\vimchanhxa-datathon\data\processed\final_features.csv


---
**Kết luận:**

---
**Notebooks tiếp theo:** [05_BASELINE_MODEL_.ipynb](05_BASELINE_MODEL_.ipynb) - Xây dựng mô hình cơ sở

In [35]:
df_base.isnull().sum()

date                  0
revenue             548
cogs                548
is_test               0
order_count         548
unique_customers    548
total_quantity      548
day_of_week           0
day_of_month          0
day_of_year           0
month                 0
year                  0
is_weekend            0
days_since_start      0
is_payday             0
is_double_day         0
days_to_tet           0
is_pre_tet_rush       0
is_tet_holiday        0
is_promo              0
total_discount        0
dtype: int64